# Build the br-ac-chat image

Run this notebook from the **`app/`** folder (next to `app.py` and `Dockerfile`) on a **SageMaker AI notebook instance** - they have Docker and the AWS CLI. In SageMaker Studio, Docker must be enabled for the domain.

It will:
1. **Create the ECR repository** and an **ECR pull-through cache** rule for ECR Public (AWS CLI; once - re-running is safe)
2. Install the app's Python packages for the target CPU (like the agent notebook does)
3. **Build and push** the image. The Dockerfile has no `RUN` steps, so nothing is downloaded during the build; the Python base image comes through the pull-through cache over your ECR VPC endpoint
4. *(optional)* Point an existing `br-ac-chat` stack at the new image. For the first deploy, run `scripts/deploy.sh` with the `IMAGE_URI` printed in step 3

**Notebook execution role needs:**
- ECR: `GetAuthorizationToken`; on `repository/br-ac-chat`: `DescribeRepositories`, `CreateRepository`, `TagResource`, `PutLifecyclePolicy`, and push (`BatchCheckLayerAvailability`, `InitiateLayerUpload`, `UploadLayerPart`, `CompleteLayerUpload`, `PutImage`, `BatchGetImage`, `GetDownloadUrlForLayer`, `DescribeImages`); `DescribePullThroughCacheRules`, `CreatePullThroughCacheRule`; on `repository/br-ac-chat-public/*`: `CreateRepository`, `BatchImportUpstreamImage`, `BatchGetImage`, `GetDownloadUrlForLayer`
- KMS (if `ECR_KMS_KEY_ARN` is set): `DescribeKey`, `CreateGrant`, `GenerateDataKey`, `Decrypt` on that key
- Step 4 only: `cloudformation:DescribeStacks`, `UpdateStack`, `DescribeStackEvents`, plus `iam:PassRole` on the stack's roles (or a CloudFormation service role)

## 0. Configuration

In [ ]:
import os, sys, json, time, platform
from pathlib import Path
import boto3

PROJECT         = "br-ac-chat"        # ECR repository name and CloudFormation stack name
REGION          = boto3.Session().region_name or "ca-central-1"
TARGET_ARCH     = "auto"              # "auto" (this notebook's CPU), "arm64" or "amd64"
ECR_KMS_KEY_ARN = ""                  # CMK for the repository; blank = ECR's AWS managed KMS key. Can't be changed later
DEPLOY_STACK    = True                # step 4: if the stack exists, point it at the new image

# Required tags (same values as the stacks)
TAGS = {"AppCatID": "12345", "Description": "Chat app container images", "ServiceName": "AgentCore POC",
        "Support-Team": "CLOUD", "Environment": "dev", "Project": PROJECT}

if TARGET_ARCH == "auto":
    TARGET_ARCH = "arm64" if platform.machine() in ("aarch64", "arm64") else "amd64"
ACCOUNT  = boto3.client("sts", region_name=REGION).get_caller_identity()["Account"]
REGISTRY = f"{ACCOUNT}.dkr.ecr.{REGION}.amazonaws.com"
IMAGE_TAG = time.strftime("v%Y%m%d-%H%M%S")          # tags are immutable - a new one each build

env = {
    "AWS_DEFAULT_REGION": REGION,
    "PROJECT": PROJECT,
    "REGISTRY": REGISTRY,
    "REPO_URI": f"{REGISTRY}/{PROJECT}",
    "IMAGE_URI": f"{REGISTRY}/{PROJECT}:{IMAGE_TAG}",
    "CACHE_PREFIX": f"{PROJECT}-public",
    "BASE_IMAGE": f"{REGISTRY}/{PROJECT}-public/docker/library/python:3.12-slim",
    "TARGET_ARCH": TARGET_ARCH,
    "CPU_ARCHITECTURE": {"arm64": "ARM64", "amd64": "X86_64"}[TARGET_ARCH],   # the stack's CpuArchitecture
    "WHEEL_PLATFORMS": {"arm64": "manylinux2014_aarch64 manylinux_2_28_aarch64",
                        "amd64": "manylinux2014_x86_64 manylinux_2_28_x86_64"}[TARGET_ARCH],
    "ECR_KMS_KEY_ARN": ECR_KMS_KEY_ARN,
    "ECR_TAGS_JSON": json.dumps([{"Key": k, "Value": v} for k, v in TAGS.items()]),
    "PYTHON": sys.executable,
    "APP_DIR": str(Path.cwd()),
    "CONTEXT": str(Path.cwd() / "build" / "context"),
}
os.environ.update(env)      # the %%bash cells below read these

for f in ("app.py", "entrypoint.py", "requirements.txt", "Dockerfile", "streamlit_config/config.toml", "run/README.txt"):
    assert (Path.cwd() / f).exists(), f"{f} not found - open this notebook from the app/ folder"
for k in ("AWS_DEFAULT_REGION", "REPO_URI", "IMAGE_URI", "BASE_IMAGE", "TARGET_ARCH", "CPU_ARCHITECTURE"):
    print(f"{k:18s} {env[k]}")

## 1. Create the ECR repository and pull-through cache (AWS CLI)

Creates them only if they don't exist yet, so re-running is safe.

- **Repository `br-ac-chat`:** KMS-encrypted (your CMK if `ECR_KMS_KEY_ARN` is set), immutable tags, scan on push, keeps the last 10 images, required tags.
- **Pull-through cache rule `br-ac-chat-public` → `public.ecr.aws`:** ECR fetches public images on AWS's side, so `docker pull <account>.dkr.ecr.<region>.amazonaws.com/br-ac-chat-public/docker/library/python:3.12-slim` works through your `ecr.dkr` VPC endpoint with no internet access.

In [ ]:
%%bash
set -euo pipefail
aws --version

if aws ecr describe-repositories --repository-names "$PROJECT" >/dev/null 2>&1; then
  echo "Repository $PROJECT already exists"
else
  ENCRYPTION="encryptionType=KMS"
  [[ -n "$ECR_KMS_KEY_ARN" ]] && ENCRYPTION="encryptionType=KMS,kmsKey=$ECR_KMS_KEY_ARN"
  aws ecr create-repository \
    --repository-name "$PROJECT" \
    --image-tag-mutability IMMUTABLE \
    --image-scanning-configuration scanOnPush=true \
    --encryption-configuration "$ENCRYPTION" \
    --tags "$ECR_TAGS_JSON" \
    --query "repository.repositoryUri" --output text
fi

aws ecr put-lifecycle-policy --repository-name "$PROJECT" --query "repositoryName" --output text \
  --lifecycle-policy-text '{"rules":[{"rulePriority":1,"description":"Keep the last 10 images","selection":{"tagStatus":"any","countType":"imageCountMoreThan","countNumber":10},"action":{"type":"expire"}}]}'

if aws ecr describe-pull-through-cache-rules --ecr-repository-prefixes "$CACHE_PREFIX" >/dev/null 2>&1; then
  echo "Pull-through cache rule $CACHE_PREFIX already exists"
else
  aws ecr create-pull-through-cache-rule \
    --ecr-repository-prefix "$CACHE_PREFIX" \
    --upstream-registry-url public.ecr.aws \
    --query "ecrRepositoryPrefix" --output text
fi

## 2. Install the Python packages for the target CPU

In [ ]:
%%bash
set -euo pipefail
docker version --format "Docker server {{.Server.Version}}" || {
  echo "Docker isn't available here - use a SageMaker notebook instance, or enable Docker for your Studio domain"; exit 1; }

rm -rf "$APP_DIR/build" && mkdir -p "$CONTEXT/packages"
PLATFORM_ARGS=""
for p in $WHEEL_PLATFORMS; do PLATFORM_ARGS="$PLATFORM_ARGS --platform $p"; done

# Binary wheels only, for Python 3.12 on the target CPU (same approach as the agent build)
"$PYTHON" -m pip install --quiet -r "$APP_DIR/requirements.txt" --target "$CONTEXT/packages" \
  --python-version 3.12 --implementation cp --only-binary=:all: --upgrade $PLATFORM_ARGS

# Not needed in the image: console scripts (their shebang points at this kernel) and bytecode
rm -rf "$CONTEXT/packages/bin"
find "$CONTEXT/packages" -name __pycache__ -prune -exec rm -rf {} +

WRONG=$([[ "$TARGET_ARCH" == arm64 ]] && echo x86_64 || echo aarch64)
if find "$CONTEXT/packages" -name "*.so" | grep -q "$WRONG"; then
  echo "ERROR: $WRONG binaries in the package"; exit 1
fi
for m in streamlit boto3 botocore cryptography; do
  [[ -d "$CONTEXT/packages/$m" ]] || { echo "ERROR: missing module $m"; exit 1; }
done
echo "Packages OK for $TARGET_ARCH: $(find "$CONTEXT/packages" -name '*.so' | wc -l) native extensions, $(du -sh "$CONTEXT/packages" | cut -f1)"

## 3. Build and push the image

The Dockerfile only copies files in, so an arm64 image can also be built on an x86_64 notebook.

In [ ]:
%%bash
set -euo pipefail
cp "$APP_DIR"/{app.py,entrypoint.py,Dockerfile} "$CONTEXT/"
cp -r "$APP_DIR/streamlit_config" "$APP_DIR/run" "$CONTEXT/"

aws ecr get-login-password | docker login --username AWS --password-stdin "$REGISTRY"

build() {
  docker build --platform "linux/$TARGET_ARCH" --build-arg "BASE_IMAGE=$BASE_IMAGE" -t "$IMAGE_URI" "$CONTEXT"
}
if ! DOCKER_BUILDKIT=1 build 2>&1 | tee /tmp/br-ac-chat-build.log; then
  if grep -qi buildx /tmp/br-ac-chat-build.log; then
    echo "BuildKit isn't installed here - retrying with the classic builder"
    DOCKER_BUILDKIT=0 build
  else
    echo "docker build failed. If it couldn't pull the base image, check the ecr.api / ecr.dkr / s3 endpoints"
    echo "and the role's CreateRepository + BatchImportUpstreamImage on repository/$CACHE_PREFIX/*"
    exit 1
  fi
fi

ARCH=$(docker image inspect "$IMAGE_URI" --format "{{.Architecture}}")
[[ "$ARCH" == "$TARGET_ARCH" ]] || { echo "ERROR: image is $ARCH, expected $TARGET_ARCH"; exit 1; }

docker push "$IMAGE_URI"
echo
echo "Pushed. For the stack:"
echo "  IMAGE_URI=$IMAGE_URI"
echo "  CPU_ARCHITECTURE=$CPU_ARCHITECTURE"

## 4. *(optional)* Point the stack at the new image

**First time:** the stack doesn't exist yet - run `scripts/deploy.sh` with the two values printed above:

```bash
IMAGE_URI=<from step 3> CPU_ARCHITECTURE=<from step 3> ./scripts/deploy.sh
```

**After that:** with `DEPLOY_STACK = True` this updates the existing stack, changing only `ImageUri` and `CpuArchitecture`. The service rolls onto the new image; a failed rollout rolls back automatically.

In [ ]:
cfn = boto3.client("cloudformation", region_name=REGION)
try:
    stack = cfn.describe_stacks(StackName=PROJECT)["Stacks"][0]
except cfn.exceptions.ClientError:
    stack = None

if not stack:
    print(f"Stack {PROJECT} doesn't exist yet. Deploy it with:\n"
          f"  IMAGE_URI={env['IMAGE_URI']} CPU_ARCHITECTURE={env['CPU_ARCHITECTURE']} ./scripts/deploy.sh")
elif not DEPLOY_STACK:
    print(f"Skipped. Update the stack with ImageUri={env['IMAGE_URI']} CpuArchitecture={env['CPU_ARCHITECTURE']}")
else:
    new = {"ImageUri": env["IMAGE_URI"], "CpuArchitecture": env["CPU_ARCHITECTURE"]}
    params = [{"ParameterKey": p["ParameterKey"], "ParameterValue": new[p["ParameterKey"]]}
              if p["ParameterKey"] in new else {"ParameterKey": p["ParameterKey"], "UsePreviousValue": True}
              for p in stack["Parameters"]]
    cfn.update_stack(StackName=PROJECT, UsePreviousTemplate=True, Parameters=params,
                     Capabilities=["CAPABILITY_IAM"])
    print(f"Updating {PROJECT} to {env['IMAGE_URI']} - waiting for the service to become healthy ...")
    try:
        cfn.get_waiter("stack_update_complete").wait(
            StackName=PROJECT, WaiterConfig={"Delay": 15, "MaxAttempts": 120})
    except Exception as e:
        print("Update did not complete:", e)
        for ev in cfn.describe_stack_events(StackName=PROJECT)["StackEvents"][:15]:
            if "FAILED" in ev["ResourceStatus"]:
                print(f"  {ev['LogicalResourceId']}: {ev.get('ResourceStatusReason')}")
        raise
    outs = {o["OutputKey"]: o["OutputValue"]
            for o in cfn.describe_stacks(StackName=PROJECT)["Stacks"][0]["Outputs"]}
    print(f"\nDone. Open {outs['AppUrl']}  (self-signed certificate - expect a browser warning)")

## 5. *(optional)* Free disk space on this notebook

In [ ]:
%%bash
docker rmi "$IMAGE_URI" || true
rm -rf "$APP_DIR/build"